In [1]:
import warnings

warnings.filterwarnings('ignore') 

In [2]:
ensemble_dir = "../qgis_tifs/ensembles"
out_csv = "comparacao_baselines.csv"

IBGE_DATA = {
    "Taiobeiras": 33629.5,
    "Santana_do_Paraiso": 7167.4,
    "Belo_Oriente": 8408.4,
    "Carbonita": 35746.6,
    "Sao_Joao_do_Paraiso": 44185.2,
    "Ipaba": 2453.8,
    "Josenopolis": 11345.5,
    "Veredinha": 12602.2,
    "Minas_Novas": 35104.8,
    "Itamarandiba": 52205.0
}

INDEPENDENT_DATA = {
    "Taiobeiras": 16550.0,
    "Santana_do_Paraiso": 7000.0,
    "Belo_Oriente": 9750.0,
    "Carbonita": 39500.0,
    "Sao_Joao_do_Paraiso": 48001.0,
    "Ipaba": 2650.0,
    "Josenopolis": 7650.0,
    "Veredinha": 12860.0,
    "Minas_Novas": 37057.0,
    "Itamarandiba": 78300.0
}


In [3]:
import glob
import os

tifs = sorted(glob.glob(os.path.join(ensemble_dir, "*.tif")))

In [4]:
import numpy as np
import pandas as pd
import rasterio


results = []

for tif_path in tifs:
    filename = os.path.basename(tif_path)
    city_name = None

    for key in IBGE_DATA.keys():
        if key in filename:
            city_name = key
            break

    config_name = (filename.replace(f"{city_name}_", "").replace(".tif", ""))

    with rasterio.open(tif_path) as src:
        mask = src.read(1)
        res_x, res_y = src.res

        pixel_area_m2 = res_x * res_y
        positive_pixels = np.count_nonzero(mask == 1)

        predicted_area_ha = (positive_pixels * pixel_area_m2) / 10000.0

    ibge_area_ha = IBGE_DATA[city_name]
    independent_area_ha = INDEPENDENT_DATA.get(city_name, np.nan)

    diff_ibge = (predicted_area_ha - ibge_area_ha)
    erro_ibge = (diff_ibge / ibge_area_ha) * 100

    if pd.notna(independent_area_ha):
        diff_independent = (predicted_area_ha - independent_area_ha)
        erro_independent = (diff_independent / independent_area_ha) * 100

    else:
        diff_independent = np.nan
        erro_independent = np.nan

    results.append({"Cidade": city_name.replace("_", " "),
                    "Configuração": config_name,
                    "Área IBGE (ha)": round(ibge_area_ha, 2),
                    "Área Independente (ha)": (round(independent_area_ha, 2) if pd.notna(independent_area_ha) else np.nan),
                    "Área Predita (ha)": round(predicted_area_ha, 2),
                    "Diferença vs IBGE (ha)": round(diff_ibge, 2),
                    "Erro vs IBGE (%)": round(erro_ibge, 2),
                    "Diferença vs Independente (ha)": (round(diff_independent, 2) if pd.notna(diff_independent) else np.nan),
                    "Erro vs Independente (%)": (round(erro_independent, 2) if pd.notna(erro_independent) else np.nan)
                    })


df = pd.DataFrame(results)
df = df.sort_values(by=["Cidade", "Configuração"])

display(df)

,Cidade,Configuração,Área IBGE (ha),Área Independente (ha),Área Predita (ha),Diferença vs IBGE (ha),Erro vs IBGE (%),Diferença vs Independente (ha),Erro vs Independente (%)
0,Belo Oriente,aggregate_2d,8408.4,9750.0,9815.73,1407.33,16.74,65.73,0.67
1,Belo Oriente,aggregate_3d,8408.4,9750.0,10425.35,2016.95,23.99,675.35,6.93
2,Belo Oriente,aggregate_all,8408.4,9750.0,10769.93,2361.53,28.09,1019.93,10.46
3,Belo Oriente,majority_2d,8408.4,9750.0,9815.73,1407.33,16.74,65.73,0.67
4,Belo Oriente,majority_3d,8408.4,9750.0,9331.22,922.82,10.97,-418.78,-4.30
5,Belo Oriente,majority_all,8408.4,9750.0,9504.42,1096.02,13.03,-245.58,-2.52
6,Carbonita,aggregate_2d,35746.6,39500.0,39866.13,4119.53,11.52,366.13,0.93
7,Carbonita,aggregate_3d,35746.6,39500.0,42129.42,6382.82,17.86,2629.42,6.66
8,Carbonita,aggregate_all,35746.6,39500.0,42573.10,6826.50,19.10,3073.10,7.78
9,Carbonita,majority_2d,35746.6,39500.0,39866.13,4119.53,11.52,366.13,0.93


In [5]:
df.to_csv(out_csv, index=False)